# Aerial OBB Object Detection & Benchmark Suite
### Google Colab GPU Runner with Google Drive Persistence & Crash Resumption

Run this notebook in Google Colab with a **GPU runtime**:
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU (or A100/V100)`

This pipeline benchmarks oriented bounding box (OBB) models across aerial datasets with:
1. **Google Drive Integration**: Mounts `/content/drive/MyDrive/object-detection` for persistent storage of datasets (`data/`), model weights (`weights/`), and outputs (`results/`).
2. **Crash-Resilient Checkpointing**: Automatically checks for existing checkpoints upon restart and resumes without repeating completed work.
3. **Proactive RAM Protection**: Actively monitors memory usage, throttling batch sizes and running garbage collection to prevent Colab OOM crashes.
4. **Complete Multi-Dataset & Multi-Model Evaluation**: Evaluates models (`yolov8n-obb`, `yolo11n-obb`, `custom-obb`) across datasets (`CODrone`, `VisDrone`, `DOTA`) with comprehensive metrics (mAP50, mAP75, mAP50-95, Precision, Recall, F1, Accuracy, Angle MAE, Pearson r, R², FPS) and visual diagnostic heatmaps.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Initialize Google Drive directory structure for object-detection
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/object-detection")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_RESULTS = DRIVE_ROOT / "results"

for folder in [DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir : {DRIVE_DATA}")
print(f"    - Weights Dir  : {DRIVE_WEIGHTS}")
print(f"    - Results Dir  : {DRIVE_RESULTS}")

In [ ]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Suyash8/minor-project.git"
if not os.path.exists("/content/minor-project"):
    !git clone $REPO_URL /content/minor-project
%cd /content/minor-project
!git pull

In [ ]:
# 5. Install required packages (Ultralytics, Shapely, Scipy, etc.)
!python install.py

In [ ]:
# 6. Optional: Dataset Linking / Setup Guide
# If your datasets (codrone, visdrone, dota) are stored in Google Drive at:
# /content/drive/MyDrive/object-detection/data/
# the pipeline automatically finds them!
#
# To check dataset readiness:
import os
for d in ["codrone", "visdrone", "dota"]:
    drive_path = f"/content/drive/MyDrive/object-detection/data/{d}"
    local_path = f"data/{d}"
    exists_drive = os.path.exists(drive_path)
    exists_local = os.path.exists(local_path)
    print(f"Dataset '{d}': Drive={exists_drive}, Local={exists_local}")

In [ ]:
# 7. Fast Smoke Test (CPU test mode verifying pre-eval, training, post-eval, deltas, and plots in seconds)
!python scripts/run_pipeline.py \
    --datasets visdrone codrone dota \
    --models yolov8n-obb yolo11n-obb custom-obb \
    --mode full \
    --test \
    --save-plots \
    --run-name smoke_test_colab


In [ ]:
# 8. Full End-to-End Pipeline on GPU:
# Stage 1: Pre-Training Baseline Evaluation (measures zero-shot / base weights)
# Stage 2: GPU Training & Fine-Tuning Suite (saturates T4 GPU with FP16 AMP & multi-worker loaders)
# Stage 3: Post-Training Evaluation (measures fine-tuned weights on validation set)
# Stage 4: Comparative Delta Synthesis (computes exact empirical gains: ΔmAP50, ΔF1, ΔAngle MAE)
#
# Checkpoints and plots automatically persist in Google Drive (/content/drive/MyDrive/object-detection/)
# If interrupted or disconnected, re-running this cell automatically resumes without repeating work!
!python scripts/run_pipeline.py \
    --datasets visdrone codrone dota \
    --models yolov8n-obb yolo11n-obb custom-obb \
    --mode full \
    --epochs 15 \
    --train-batch-size 16 \
    --train-workers 4 \
    --batch-size 8 \
    --device cuda \
    --save-plots \
    --resume True \
    --run-name full_train_and_benchmark


In [ ]:
# 9. Display Generated Benchmark Report & Comparative Progression Plots Inline
import os
import glob
from pathlib import Path
from IPython.display import Image, display, Markdown

# Search in Google Drive results first, then local fallback
search_dirs = ["/content/drive/MyDrive/object-detection/results/*", "results/*"]
run_dirs = []
for pattern in search_dirs:
    for p in glob.glob(pattern):
        if os.path.isdir(p) and not p.endswith(".tmp"):
            run_dirs.append(p)

run_dirs = sorted(set(run_dirs))
if run_dirs:
    latest_run = run_dirs[-1]
    print(f"Latest Run Directory: {latest_run}")
    report_file = os.path.join(latest_run, "benchmark_report.md")
    if os.path.exists(report_file):
        with open(report_file) as f:
            display(Markdown(f.read()))

    # 1. Display Before-vs-After Gain Progression Chart
    pre_post_chart = os.path.join(latest_run, "plots", "pre_vs_post_comparison.png")
    if os.path.exists(pre_post_chart):
        print("\n=======================================================")
        print("--- Pre-Training vs Post-Training Performance Gain ---")
        print("=======================================================")
        display(Image(filename=pre_post_chart))

    # 2. Display Overall Model Benchmark Comparison Chart
    comparison_chart = os.path.join(latest_run, "plots", "model_benchmark_comparison.png")
    if os.path.exists(comparison_chart):
        print("\n--- Overall Model Benchmark Comparison ---")
        display(Image(filename=comparison_chart))

    # 3. Display Confusion Matrices and Correlation Plots
    for img_path in sorted(glob.glob(f"{latest_run}/plots/*.png")):
        bname = os.path.basename(img_path)
        if bname not in ("model_benchmark_comparison.png", "pre_vs_post_comparison.png"):
            print(f"\nDisplaying: {bname}")
            display(Image(filename=img_path))
else:
    print("No results directory found yet. Run step 8 above first.")
